In [1]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import numpy as np
import math
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import pandas as pd


In [2]:
!pip install pytorch-ignite
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping


/home/adminnio/miniconda3/envs/navigator/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [3]:
class SineLayer(nn.Module):
    def __init__(self,w0):
        super(SineLayer, self).__init__()
        self.w0 = w0

    def forward(self, x):
        return torch.sin(self.w0 * x)

class Model(nn.Module):
    def __init__(self,w0=30,in_features=2,out_features=3,hidden = 256):
        super(Model, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), SineLayer(w0),
            nn.Linear(hidden, hidden),SineLayer(w0),
            nn.Linear(hidden, 512),SineLayer(w0),
            nn.Linear(512, 512),SineLayer(w0),
            nn.Linear(512, 512),SineLayer(w0),
            nn.Linear(512, 512),SineLayer(w0),
            nn.Linear(512, hidden),SineLayer(w0),
            nn.Linear(hidden, hidden),SineLayer(w0),
            nn.Linear(hidden, out_features)
                 )

        with torch.no_grad():
            self.net[0].weight.uniform_(-1. / in_features, 1. / in_features)
            self.net[2].weight.uniform_(-np.sqrt(6. / hidden) / w0, np.sqrt(6. / hidden) / w0)
            self.net[4].weight.uniform_(-np.sqrt(6. / 512) / w0, np.sqrt(6. / 512) / w0)
            self.net[6].weight.uniform_(-np.sqrt(6. / 512) / w0, np.sqrt(6. / 512) / w0)
            self.net[8].weight.uniform_(-np.sqrt(6. / 512) / w0, np.sqrt(6. / 512) / w0)
            self.net[10].weight.uniform_(-np.sqrt(6. / 512) / w0, np.sqrt(6. / 512) / w0)
            self.net[12].weight.uniform_(-np.sqrt(6. / hidden) / w0, np.sqrt(6. / hidden) / w0)
            self.net[14].weight.uniform_(-np.sqrt(6. / hidden) / w0, np.sqrt(6. / hidden) / w0)
            self.net[16].weight.uniform_(-np.sqrt(6. / hidden) / w0, np.sqrt(6. / hidden) / w0)
        # with torch.no_grad():
        #     self.net[0].weight.uniform_(-np.sqrt(6. / hidden*w0*w0) , np.sqrt(6. / hidden*w0*w0))
        #     self.net[2].weight.uniform_(-np.sqrt(6. / hidden*w0*w0) , np.sqrt(6. / hidden*w0*w0) )
        #     self.net[4].weight.uniform_(-np.sqrt(6. / 512*w0*w0) , np.sqrt(6. / 512*w0*w0) )
        #     self.net[6].weight.uniform_(-np.sqrt(6. / 512*w0*w0) , np.sqrt(6. / 512*w0*w0) )
        #     self.net[8].weight.uniform_(-np.sqrt(6. / 512*w0*w0) , np.sqrt(6. / 512*w0*w0) )
        #     self.net[10].weight.uniform_(-np.sqrt(6. / 512*w0*w0) , np.sqrt(6. / 512*w0*w0) )
        #     self.net[12].weight.uniform_(-np.sqrt(6. / hidden*w0*w0) , np.sqrt(6. / hidden*w0*w0) )
        #     self.net[14].weight.uniform_(-np.sqrt(6. / hidden*w0*w0) , np.sqrt(6. / hidden*w0*w0) )
        #     self.net[16].weight.uniform_(-np.sqrt(6. / hidden*w0*w0) , np.sqrt(6. / hidden*w0*w0) )

    def forward(self,x):
        return self.net(x)
                

In [4]:
class ImageDataloadr(Dataset):
    def __init__(self):
        path = "/media/adminnio/Volume/Data_NerfRaw/Lakshwadeep/Known_Lak/1/sp+lg_radial/images/G0163545.JPG"
        
        # Load and resize image
        image = Image.open(path).convert('RGB')
        image = image.resize((512, 512))  # Resize before converting to NumPy
        image_np = np.array(image).astype(np.float32)   # Normalize to [0, 1]

        # Get height and width
        self.height, self.width, _ = image_np.shape

        # Create coordinate grid
        x = np.linspace(-1, 1, self.width, dtype=np.float32)
        y = np.linspace(-1, 1, self.height, dtype=np.float32)
        cols, rows = np.meshgrid(x, y, indexing='ij')  # x=width, y=height
        
        self.coords = np.stack((rows, cols), axis=-1).reshape(-1, 2)
        self.rgb_values = image_np.reshape(-1, 3).astype(np.float32)   
        self.n_samples = self.height * self.width

    def __getitem__(self, index):
        coord = torch.tensor(self.coords[index], dtype=torch.float32)
        rgb = torch.tensor(self.rgb_values[index], dtype=torch.float32)
        return coord, rgb

    def __len__(self):
        return self.n_samples


In [5]:
dataset = ImageDataloadr()
dataloader = DataLoader(dataset=dataset, batch_size=2048, shuffle=True,num_workers=0)
dataiter = iter(dataloader)
data = next(dataiter)
f,l =data

print(l)

tensor([[ 71., 176., 187.],
        [ 35.,  41.,  41.],
        [108.,  99.,  75.],
        ...,
        [197., 188., 159.],
        [ 43.,  43.,  41.],
        [ 90.,  85.,  71.]])


In [6]:
np.shape(f)

torch.Size([2048, 2])

In [7]:
def Lsaturation(J_hat):
    # J_hat has shape [H, W, 3] (H: height, W: width, 3 channels)
    N = J_hat.shape[0] * J_hat.shape[1]  # H * W (total number of pixels)
    total = 0
    for c in range(3):
        # Channel-wise: clamp to zero if greater than 1
        diff = torch.clamp(J_hat[:, :, c] - 1, min=0)
        total += torch.sum(diff ** 2)
    return total / (3 * N)

def intensity(J_hat):
    N = J_hat.shape[0] * J_hat.shape[1]  # H * W (total number of pixels)
    total = 0
    for c in range(3):
        # Compute the mean difference per channel
        mean_diff = torch.sum(J_hat[:, :, c] - 0.5) / N
        total += mean_diff ** 2
    return total / 3

In [14]:

def visualize_learned_image(model, height, width, device, epoch):
    # Create a mesh grid of pixel coordinates (0 to height-1, 0 to width-1)
    x = torch.linspace(-1, 1, width, device=device)
    y = torch.linspace(-1, 1, height, device=device)
    cols, rows = torch.meshgrid(x, y, indexing='ij')  # Match dataloader's coordinate order
    coords = torch.stack((rows, cols), dim=-1).reshape(-1, 2).to(torch.float32)

    with torch.no_grad():
        output = model(coords)  # Directly use normalized coordinates
    # Since the model outputs values in the range [0, 255], we scale them to [0, 1]
    rgb = (output / 255.0).clamp(0, 1).reshape(512, 512, 3).cpu().numpy()
    
    # Plot the learned image
        # Plot and save
    plt.imshow(rgb)
    plt.title(f"Learned Image - Epoch {epoch}")
    plt.axis('off')
    rgb_uint8 = (rgb * 255).astype('uint8')
    img = Image.fromarray(rgb_uint8)
    save_path = f"trainingout/learned_image_epoch_{epoch:04d}.png"
    img.save(save_path)

In [ ]:
from tqdm import tqdm
import torch.nn as nn


total_batches = 10
epochs = 2000 
model = Model(in_features=2, out_features=3).to(device)
criterion = nn.MSELoss()
learning_rate = 5e-5
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,factor =0.1, patience=5,verbose=True)
batches_per_epoch = len(dataloader)
total_steps = epochs* batches_per_epoch  
losses = []
with tqdm(total=total_steps, desc="Total Training Progress") as pbar:
    for epoch in range(epochs):
        for features, labels in dataloader:
            features = features.to(torch.float32).to(device)
            labels = labels.to(torch.float32).to(device)

            # Forward pass
            scores = model(features)
            loss = criterion(scores, labels)
            losses.append(loss)
            # Backward pass             l1.append
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Update the single global progress bar
            pbar.set_postfix(loss=loss.item())
            pbar.update(1)
        scheduler.step(loss)
        # This block now runs once per epoch, every 500th epoch
        if epoch % 10 == 0:
            with torch.no_grad():
                output = model(features)
                visualize_learned_image(model, dataloader.dataset.height, dataloader.dataset.width, device=device, epoch=epoch)
        elif epoch % 2 == 0 and epoch <= 50:
            with torch.no_grad():
                visualize_learned_image(model, dataloader.dataset.height, dataloader.dataset.width,device=device, epoch=epoch)


Total Training Progress:   1%|▏                          | 1949/256000 [00:32<1:27:13, 48.54it/s, loss=1.11e+4]